In [ ]:
# 导入所需库
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler  # 决策树对尺度不敏感，此处仅为统一流程保留

# ===================== 1. 加载并预处理数据集 =====================
# 加载鸢尾花数据集
iris = load_iris()
# 转换为DataFrame，方便数据查看和处理
data = pd.DataFrame(data=iris.data, columns=iris.feature_names)
data['target'] = iris.target  # 添加目标标签列

# 决策树算法对特征尺度不敏感，无需标准化（此处注释保留标准化代码，便于对比其他算法）
# scaler = StandardScaler()
# data[iris.feature_names] = scaler.fit_transform(data[iris.feature_names])

# 划分特征矩阵X和目标向量y
X = data[iris.feature_names]  # 4个特征：花萼长/宽、花瓣长/宽
y = data['target']            # 3个类别：0=setosa, 1=versicolor, 2=virginica

# 按7:3划分训练集和测试集，固定随机种子保证结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 打印数据集基本信息
print("=== 数据集基本信息 ===")
print(f"训练集样本数: {X_train.shape[0]}, 测试集样本数: {X_test.shape[0]}")
print(f"特征列: {iris.feature_names}")
print(f"类别名称: {iris.target_names}\n")

# ===================== 2. 初始化并训练基础决策树模型 =====================
# 初始化决策树分类器（默认参数：基尼系数划分，最大深度不限）
dt = DecisionTreeClassifier(random_state=42)
# 用训练集拟合模型
dt.fit(X_train, y_train)

# 对测试集进行预测
y_pred = dt.predict(X_test)

# ===================== 3. 基础模型评估 =====================
print("=== 基础决策树模型评估结果 ===")
# 计算并打印准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"测试集准确率: {accuracy:.4f}")

# 打印分类报告（精确率、召回率、F1分数）
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 打印混淆矩阵
print("\n混淆矩阵:")
print(confusion_matrix(y_test, y_pred))

# ===================== 4. 决策树可视化（核心优势：可解释性） =====================
plt.figure(figsize=(15, 10))  # 设置画布大小，保证树结构清晰
# 绘制决策树
plot_tree(
    dt,
    feature_names=iris.feature_names,  # 特征名称
    class_names=iris.target_names,     # 类别名称
    filled=True,                       # 填充颜色（按类别区分）
    rounded=True,                      # 圆角矩形
    fontsize=10                        # 字体大小
)
plt.title("鸢尾花数据集决策树分类模型", fontsize=16)
plt.savefig("iris_decision_tree.png", dpi=300, bbox_inches='tight')  # 保存图片
plt.show()

# ===================== 5. 网格搜索优化决策树参数（避免过拟合） =====================
# 定义参数搜索范围（重点优化易导致过拟合的参数）
param_grid = {
    'criterion': ['gini', 'entropy'],  # 划分准则：基尼系数/信息熵
    'max_depth': [2, 3, 4, 5, None],   # 树的最大深度（None表示不限）
    'min_samples_split': [2, 5, 10],   # 节点分裂所需最小样本数
    'min_samples_leaf': [1, 2, 5]      # 叶节点所需最小样本数
}

# 初始化网格搜索（5折交叉验证，按准确率评分）
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1  # 利用所有CPU核心加速
)

# 训练并搜索最优参数
grid_search.fit(X_train, y_train)

# 打印最优参数和交叉验证最优准确率
print("\n=== 网格搜索最优参数 ===")
print(f"最优参数组合: {grid_search.best_params_}")
print(f"交叉验证最优准确率: {grid_search.best_score_:.4f}")

# ===================== 6. 最优模型评估与预测 =====================
# 获取最优模型
dt_best = grid_search.best_estimator_
# 用最优模型预测测试集
y_pred_best = dt_best.predict(X_test)

# 评估最优模型
print("\n=== 最优决策树模型评估结果 ===")
print(f"测试集准确率（最优模型）: {accuracy_score(y_test, y_pred_best):.4f}")

# 新样本预测（模拟实际应用场景）
print("\n=== 新样本预测 ===")
# 定义2个新样本（特征顺序：花萼长、花萼宽、花瓣长、花瓣宽）
new_samples = [
    [5.1, 3.5, 1.4, 0.2],  # 典型setosa样本
    [6.7, 3.0, 5.2, 2.3]   # 典型virginica样本
]
# 转换为DataFrame（保证特征名称匹配）
new_samples_df = pd.DataFrame(new_samples, columns=iris.feature_names)

# 最优模型预测
predictions = dt_best.predict(new_samples_df)
# 预测类别概率（可选，展示模型置信度）
predict_proba = dt_best.predict_proba(new_samples_df)

# 打印预测结果
for i, (sample, pred, prob) in enumerate(zip(new_samples, predictions, predict_proba)):
    print(f"新样本{i+1}: {sample}")
    print(f"预测类别: {pred} ({iris.target_names[pred]})")
    print(f"类别概率: {dict(zip(iris.target_names, prob.round(4)))}")
    print("-" * 50)

# ===================== 7. 特征重要性分析 =====================
print("\n=== 特征重要性分析 ===")
# 提取特征重要性（决策树的核心属性，体现特征对分类的贡献）
feature_importance = pd.DataFrame({
    '特征名称': iris.feature_names,
    '重要性': dt_best.feature_importances_
}).sort_values(by='重要性', ascending=False)

print(feature_importance)
# 可视化特征重要性
plt.figure(figsize=(10, 6))
plt.bar(feature_importance['特征名称'], feature_importance['重要性'])
plt.title("决策树模型特征重要性", fontsize=14)
plt.xlabel("特征")
plt.ylabel("重要性")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()